# Experiment 7d — Pure-DL hybrid: gated MLP + attention encoder over past target

Standalone deep-learning model. **Not** an ensemble with LGBM — the ensemble (`7c`) is now the leaderboard baseline at WAPE 14.68, and this notebook tests whether a single neural network with learned feature selection can match or beat it.

Two professor-driven constraints shape this architecture:

1. *"The model should figure out feature selection itself."* → **Learnable per-feature gates** (sigmoid scalar per covariate, multiplied into the input). The headline output of this notebook is the table of learned gate values per feature, averaged across seeds.
2. *"The underlying task is sequence-to-sequence translation."* → **Past encoder**: a small transformer that consumes the last 168 hours of `(past_covariates + past_target + past_time)` for each series and produces a fixed-length context vector used by the per-row head. No autoregressive feedback — encoder always summarizes train-only data, even at inference, so we never recreate the compounding-error problem from experiments 1–3.

The architecture deliberately keeps the parts of the 7a-tabular MLP that already worked (per-series embedding, per-series target normalization, dense MLP head with BatchNorm+ReLU+Dropout) and adds *only* the two professor-required ingredients on top.

Decision shape:

- If solo CV-WAPE ≤ **14.52** (the 7a-tabular MLP) → the hybrid extracted real signal from past target + gating. Ship as the pure-DL submission candidate.
- If 14.52 < solo CV-WAPE ≤ 14.78 (LGBM) → the additions didn't help vs. the simpler MLP; gate values are still the deliverable, but the simpler MLP is the better pure-DL submission.
- If > 14.78 → architecture is broken or undertrained. Investigate before iterating.

---
## §0. Configuration

Paths and feature config mirror exp-7a-tabular §0 *for the parts that must match* (so the LGBM anchor in §3 lands at 14.78 and the residual-correlation diagnostic in §7 is comparable). Adds hybrid-specific hyperparameters in the second block. Decision thresholds in the third block are stated **before any training runs**.

In [4]:
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR     = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR   = Path("/kaggle/working")
# Local:
# DATA_DIR   = Path("data")
# OUTPUT_DIR = Path("experiment-7")

TRAIN_PATH     = DATA_DIR / "train.csv"
VAL_INPUT_PATH = DATA_DIR / "validation_input.csv"
VAL_INDEX_PATH = DATA_DIR / "forecast_index_validation.csv"

# ── Reproducibility / CV ─────────────────────────────────────────────────────
SEED         = 42
N_FOLDS      = 3
EVAL_HORIZON = 336

# ── Feature config (anchor LGBM only — the hybrid feeds on raw past sequence,
#    not on engineered lag/lead/rolling features) ─────────────────────────────
DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
TIME_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos",
             "doy_sin", "doy_cos", "is_weekend"]
TOP_DOM   = ["queue_pressure_forecast", "network_pressure_forecast",
             "workload_intensity", "shock_risk", "event_load_forecast"]
TOP3_ROLL = TOP_DOM[:3]
LAGS_DOM, LEADS_DOM, ROLL_WINS = [1, 24], [1, 24], [6, 24]

BASELINE_LGBM = dict(
    n_estimators=600, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
    reg_lambda=1.0, verbose=-1,
)

# ── Hybrid model config ──────────────────────────────────────────────────────
L_PAST            = 168          # 168h = 1 week of context
LEAD_MAX          = EVAL_HORIZON # max future-step lead seen during training (matches val horizon)

# Past encoder (transformer over the L_PAST window)
ENC_HIDDEN        = 128
ENC_HEADS         = 4
ENC_LAYERS        = 2
ENC_DROPOUT       = 0.1

# Per-series embedding
EMBED_DIM         = 16

# Head MLP (matches 7a-tabular shape)
HEAD_HIDDEN       = [256, 256, 128]
HEAD_DROPOUT      = 0.2

# Training
BATCH             = 256
LR                = 1e-3
WEIGHT_DECAY      = 1e-4
EPOCHS            = 30
EARLY_STOP        = 5
WINDOWS_PER_SERIES_PER_EPOCH = 128   # subsampling → ~12,300 train samples / epoch
SEEDS             = [42, 101, 202]   # diagnostic-tier (3 seeds). Production: 5.

# ── Decision thresholds — stated up front ────────────────────────────────────
BEATS_MLP_THRESHOLD  = 14.52     # 7a-tabular MLP solo CV-WAPE
BEATS_LGBM_THRESHOLD = 14.78     # 5.1 / 6 / 7a anchor
R_STRONG_DIVERSITY   = 0.85      # decorrelation rule from 7a / 7b

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR    = {DATA_DIR}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"Hybrid arch : encoder L_past={L_PAST}, transformer {ENC_LAYERS}×{ENC_HIDDEN} h{ENC_HEADS}, head {HEAD_HIDDEN}")
print(f"Train       : batch {BATCH}, lr {LR}, epochs {EPOCHS} (early stop {EARLY_STOP}), "
      f"{WINDOWS_PER_SERIES_PER_EPOCH} windows/series/epoch, seeds {SEEDS}")
print(f"Decisions   : beat MLP @ {BEATS_MLP_THRESHOLD}, beat LGBM @ {BEATS_LGBM_THRESHOLD}, "
      f"r < {R_STRONG_DIVERSITY} for decorrelation")

DATA_DIR    = /kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data
OUTPUT_DIR  = /kaggle/working
Hybrid arch : encoder L_past=168, transformer 2×128 h4, head [256, 256, 128]
Train       : batch 256, lr 0.001, epochs 30 (early stop 5), 128 windows/series/epoch, seeds [42, 101, 202]
Decisions   : beat MLP @ 14.52, beat LGBM @ 14.78, r < 0.85 for decorrelation


---
## §1. Setup & data loading

Same as 7a-tabular §1, plus a contiguous per-series tensor representation that the past encoder needs for fast windowed slicing.

In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import RobustScaler

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

def wape(y_true, y_pred, eps=1e-8):
    return np.sum(np.abs(np.asarray(y_true) - np.asarray(y_pred))) / \
           (np.sum(np.abs(y_true)) + eps) * 100

train_df  = pd.read_csv(TRAIN_PATH,     parse_dates=["timestamp"])
val_input = pd.read_csv(VAL_INPUT_PATH, parse_dates=["timestamp"])
val_index = pd.read_csv(VAL_INDEX_PATH, parse_dates=["timestamp"])

sid_to_code = {sid: i for i, sid in enumerate(sorted(train_df["series_id"].unique()))}
N_SERIES = len(sid_to_code)

print(f"train_df: {train_df.shape}")
print(f"val_input: {val_input.shape}")
print(f"# series: {N_SERIES}")

PyTorch device: cuda
  GPU: Tesla T4
train_df: (414720, 25)
val_input: (32256, 24)
# series: 96


---
## §2. Feature engineering for the LGBM anchor + sequence prep for the hybrid

Two parallel pipelines:

- **Tabular pipeline** (only for the LGBM anchor in §3) — verbatim from exp-5.1.
- **Sequence pipeline** — per-series contiguous tensors so the encoder can grab arbitrary `[t-L_PAST : t]` windows by indexing.

In [6]:
# ── Tabular pipeline for the LGBM anchor (exp-5.1) ──────────────────────────
def impute_domain_features(df, cols):
    df = df.sort_values(["series_id", "timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(
        lambda x: x.ffill().bfill()
    )
    return df

def add_time_features(df, time_col="timestamp"):
    df = df.copy()
    t = pd.to_datetime(df[time_col])
    df["hour_sin"]   = np.sin(2 * np.pi * t.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * t.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * t.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * t.dt.dayofweek / 7)
    df["doy_sin"]    = np.sin(2 * np.pi * t.dt.dayofyear / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * t.dt.dayofyear / 365)
    df["is_weekend"] = (t.dt.dayofweek >= 5).astype(float)
    return df

def add_temporal_features(df, lag_cols, roll_cols, lags, leads, windows):
    df = df.copy().sort_values(["series_id", "timestamp"])
    new = []
    for col in lag_cols:
        g = df.groupby("series_id", sort=False)[col]
        for L in lags:
            c = f"{col}_lag{L}";  df[c] = g.shift(L);  new.append(c)
        for L in leads:
            c = f"{col}_lead{L}"; df[c] = g.shift(-L); new.append(c)
    for col in roll_cols:
        g = df.groupby("series_id", sort=False)[col]
        for W in windows:
            c_m = f"{col}_roll{W}m"
            df[c_m] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).mean())
            new.append(c_m)
            c_s = f"{col}_roll{W}s"
            df[c_s] = g.shift(1).transform(lambda x: x.rolling(W, min_periods=1).std().fillna(0))
            new.append(c_s)
    return df, new

base_tab = add_time_features(impute_domain_features(train_df, DOMAIN_FEAT_COLS))
base_tab["sid_code"] = base_tab["series_id"].map(sid_to_code).astype("int32")
base_tab, TEMPORAL_FEATS = add_temporal_features(
    base_tab, lag_cols=TOP_DOM, roll_cols=TOP3_ROLL,
    lags=LAGS_DOM, leads=LEADS_DOM, windows=ROLL_WINS,
)
NUMERIC_FEATS  = DOMAIN_FEAT_COLS + TIME_COLS + ["series_mean"] + TEMPORAL_FEATS
FEATS_WITH_SID = NUMERIC_FEATS + ["sid_code"]
print(f"LGBM-anchor features: {len(FEATS_WITH_SID)}")

# ── Sequence pipeline for the hybrid ────────────────────────────────────────
# Per-series contiguous tensor with rows = timesteps, columns =
#    [11 imputed covariates] + [7 time enc] + [target]
# Plus a per-(series, row) sid_code so we can recover sid later.
base_seq = add_time_features(impute_domain_features(train_df, DOMAIN_FEAT_COLS))
base_seq["sid_code"] = base_seq["series_id"].map(sid_to_code).astype("int32")
base_seq = base_seq.sort_values(["series_id", "timestamp"]).reset_index(drop=True)

SEQ_FEATURE_COLS = DOMAIN_FEAT_COLS + TIME_COLS + ["target"]
N_COV   = len(DOMAIN_FEAT_COLS)
N_TIME  = len(TIME_COLS)
N_FEAT_PER_STEP = N_COV + N_TIME + 1   # +1 for target
assert len(SEQ_FEATURE_COLS) == N_FEAT_PER_STEP

# Per-series tensors as a Python dict (sid_code → ndarray of shape (T_s, N_FEAT_PER_STEP))
# All series have the same length T = 4320 (verified in earlier notebooks), but we don't rely on it.
series_tensors_full = {}
for sid, code in sid_to_code.items():
    g = base_seq[base_seq["sid_code"] == code].sort_values("timestamp")
    series_tensors_full[code] = g[SEQ_FEATURE_COLS].to_numpy(dtype=np.float32)
T_PER_SERIES = next(iter(series_tensors_full.values())).shape[0]
print(f"Sequence per-series tensor shape: ({T_PER_SERIES}, {N_FEAT_PER_STEP})")
print(f"  cols: {N_COV} covariates + {N_TIME} time + 1 target = {N_FEAT_PER_STEP}")

LGBM-anchor features: 52
Sequence per-series tensor shape: (4320, 19)
  cols: 11 covariates + 7 time + 1 target = 19


---
## §3. CV framework and LGBM-MAE anchor

Same 3-fold rolling-origin CV used throughout exp-5 / exp-6 / exp-7. The LGBM-MAE baseline reproduces the 14.78 anchor and gives us OOF predictions for the residual-correlation diagnostic in §7.

In [7]:
def fold_split_tabular(end_off, length=EVAL_HORIZON):
    "For LGBM only: same as 7a-tabular."
    tr_list, te_list = [], []
    for _, g in base_tab.groupby("series_id", sort=False):
        g = g.sort_values("timestamp")
        cut = len(g) - end_off
        tr_list.append(g.iloc[:cut])
        te_list.append(g.iloc[cut:cut + length])
    tr = pd.concat(tr_list, ignore_index=True)
    te = pd.concat(te_list, ignore_index=True)
    sm = tr.groupby("series_id")["target"].mean().rename("series_mean").reset_index()
    return tr.merge(sm, on="series_id"), te.merge(sm, on="series_id")

def fold_offset(k):
    return (N_FOLDS - k) * EVAL_HORIZON

# Per-fold cut indices on the per-series sequence tensors (one int — the same number of
# tail-hours is held out per series, since all series are equal length).
def fold_train_end_idx(k):
    return T_PER_SERIES - fold_offset(k)

def fit_lgbm_predict(tr, te, feats, params, objective, categorical=None, seed=SEED):
    model = lgb.LGBMRegressor(**params, objective=objective, random_state=seed)
    fit_kw = {} if not categorical else {"categorical_feature": categorical}
    model.fit(tr[feats], tr["target"].to_numpy(float), **fit_kw)
    return model.predict(te[feats]), model

print("Running LightGBM-MAE anchor (same config as exp-5.1 / 7a)...")
oof_lgbm_rows, per_fold_lgbm = [], []
for k in range(N_FOLDS):
    tr, te = fold_split_tabular(fold_offset(k))
    preds, _ = fit_lgbm_predict(tr, te, FEATS_WITH_SID, BASELINE_LGBM,
                                "regression_l1", categorical=["sid_code"])
    per_fold_lgbm.append(wape(te["target"], preds))
    oof_lgbm_rows.append(te.assign(pred_lgbm=preds, fold=k)[
        ["series_id", "timestamp", "target", "pred_lgbm", "fold"]
    ])
oof_lgbm = pd.concat(oof_lgbm_rows, ignore_index=True)
lgbm_arr = np.array(per_fold_lgbm)

print(f"\n{'fold':>6}  {'WAPE':>8}")
for k, w in enumerate(lgbm_arr):
    print(f"  {k:>4}    {w:>7.3f}")
print(f"  ────")
print(f"  mean    {lgbm_arr.mean():>7.3f}")
print(f"  std     {lgbm_arr.std():>7.3f}")

if abs(lgbm_arr.mean() - 14.78) > 0.5:
    print(f"\n⚠️  LGBM anchor drifted from 14.78. Investigate before trusting downstream.")
else:
    print(f"\n✅ LGBM anchor reproduces — OOF predictions valid for §7 diagnostic.")

Running LightGBM-MAE anchor (same config as exp-5.1 / 7a)...

  fold      WAPE
     0     14.082
     1     15.581
     2     14.683
  ────
  mean     14.782
  std       0.616

✅ LGBM anchor reproduces — OOF predictions valid for §7 diagnostic.


---
## §4. Hybrid model — `FeatureGate` + `PastEncoder` + MLP head

Three pieces, each doing exactly one job:

- `FeatureGate(11)` — a single learnable sigmoid scalar per covariate. The model's chosen feature importance is `sigmoid(gates_logits)`; a value near 0 means "this covariate contributes nothing," near 1 means "keep at full magnitude." Initialized at logit=0 so initial gates are 0.5 (neutral).
- `PastEncoder` — 2-layer transformer over the 168-step past window. Projects each timestep `(11 covariates + 7 time + 1 target) = 19 dims` to 128 dims, adds a learned positional embedding, prepends a learnable [CLS] token, runs self-attention, returns the [CLS] vector as the per-series context.
- Head — verbatim shape of the 7a-tabular MLP `[256, 256, 128]` with BatchNorm + ReLU + Dropout. Input concatenates: `[gated current covariates (11) + current time (7) + lead_pos (1) + series embedding (16) + series_mean (1) + past_context (128)] = 164 dims`. Output: scalar in normalized space.

In [8]:
class FeatureGate(nn.Module):
    "Learnable sigmoid scalar per input feature."
    def __init__(self, n_features):
        super().__init__()
        self.gates_logits = nn.Parameter(torch.zeros(n_features))
    def forward(self, x):
        return x * torch.sigmoid(self.gates_logits)
    def values(self):
        with torch.no_grad():
            return torch.sigmoid(self.gates_logits).cpu().numpy()


class PastEncoder(nn.Module):
    "Transformer over the past window. CLS-pooled output is the per-series context vector."
    def __init__(self, n_step_features, max_len, hidden, n_heads, n_layers, dropout):
        super().__init__()
        self.in_proj = nn.Linear(n_step_features, hidden)
        self.pos     = nn.Parameter(torch.randn(1, max_len + 1, hidden) * 0.02)
        self.cls     = nn.Parameter(torch.randn(1, 1, hidden) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * 2,
            dropout=dropout, batch_first=True,
            activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.out_dim = hidden
    def forward(self, past_seq):
        # past_seq: (B, L, n_step_features)
        B, L, _ = past_seq.shape
        x = self.in_proj(past_seq)
        cls = self.cls.expand(B, 1, x.size(-1))
        x = torch.cat([cls, x], dim=1) + self.pos[:, : L + 1]
        x = self.encoder(x)
        return x[:, 0]   # (B, hidden) — CLS


class HybridModel(nn.Module):
    def __init__(self, n_cov, n_time, n_series, embed_dim,
                 enc_hidden, enc_heads, enc_layers, enc_dropout, max_past_len,
                 head_hidden, head_dropout):
        super().__init__()
        self.gate = FeatureGate(n_cov)
        self.past_enc = PastEncoder(
            n_step_features=n_cov + n_time + 1,  # cov + time + target
            max_len=max_past_len,
            hidden=enc_hidden, n_heads=enc_heads, n_layers=enc_layers,
            dropout=enc_dropout,
        )
        self.series_emb = nn.Embedding(n_series, embed_dim)

        head_in = n_cov + n_time + 1 + embed_dim + 1 + enc_hidden
        layers = []
        in_dim = head_in
        for h in head_hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h),
                       nn.ReLU(inplace=True), nn.Dropout(head_dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.head = nn.Sequential(*layers)

    def forward(self, past_seq, current_cov, current_time, lead_pos, sid, series_mean):
        # past_seq:      (B, L, n_cov + n_time + 1)
        # current_cov:   (B, n_cov)
        # current_time:  (B, n_time)
        # lead_pos:      (B, 1)
        # sid:           (B,)  int
        # series_mean:   (B, 1)
        gated = self.gate(current_cov)
        past_ctx = self.past_enc(past_seq)
        sid_e = self.series_emb(sid)
        x = torch.cat([gated, current_time, lead_pos, sid_e, series_mean, past_ctx], dim=-1)
        return self.head(x).squeeze(-1)


n_head_in = N_COV + N_TIME + 1 + EMBED_DIM + 1 + ENC_HIDDEN
print(f"Head input dim: {N_COV}(gated cov) + {N_TIME}(time) + 1(lead) + "
      f"{EMBED_DIM}(sid emb) + 1(series_mean) + {ENC_HIDDEN}(past ctx) = {n_head_in}")

Head input dim: 11(gated cov) + 7(time) + 1(lead) + 16(sid emb) + 1(series_mean) + 128(past ctx) = 164


---
## §5. Training utilities — sample generation, normalization, train loop

Two important pieces here:

- **`sample_train_indices`** — for each (series, current_t) in the fold's training slice, picks a random `t_anchor` such that `past_seq` ends just before `t_anchor` and `lead_pos = current_t − t_anchor ∈ [1, LEAD_MAX]`. This is what teaches the model to handle the seq2seq framing: every training step looks like "given an encoder window ending at T, predict the target at T + Δ." Subsampling to `WINDOWS_PER_SERIES_PER_EPOCH` keeps each epoch cheap.
- **Per-series target μ/σ + per-feature RobustScaler** — same recipe as 7a-tabular: feature scaler fit on raw (cov + time + target) columns from the fold's training slice; target predictions are computed in normalized space and un-normalized before WAPE. NaN values from edge rows are zeroed out post-scaling (the median).

Inference path is much simpler: encoder runs once per series on the fixed train-tail window, head runs `EVAL_HORIZON` times per series with varying `current_t` / `lead_pos`.

In [9]:
def fit_per_series_target_stats(tensor_by_sid, train_end_idx):
    "Per-series μ/σ from the fold's training slice's target column."
    mus, sds = {}, {}
    for sid, arr in tensor_by_sid.items():
        tgt = arr[:train_end_idx, -1]
        mus[sid] = float(np.mean(tgt))
        sds[sid] = float(max(np.std(tgt), 1e-3))
    return mus, sds

def fit_feature_scaler(tensor_by_sid, train_end_idx):
    "Per-feature RobustScaler over the fold's training slice (all series concatenated)."
    rows = np.concatenate([arr[:train_end_idx] for arr in tensor_by_sid.values()], axis=0)
    rows = np.nan_to_num(rows, nan=0.0)
    sc = RobustScaler().fit(rows)
    return sc

def apply_scalers_to_tensor(arr, feat_scaler, sid_mu, sid_sd):
    "Scale features (RobustScaler) and standardize target column per-series."
    out = feat_scaler.transform(np.nan_to_num(arr, nan=0.0)).astype(np.float32)
    # Replace the target column with the per-series standardized version,
    # computed from the *original* (pre-scaling) target values.
    raw_tgt = np.nan_to_num(arr[:, -1], nan=0.0)
    out[:, -1] = ((raw_tgt - sid_mu) / sid_sd).astype(np.float32)
    return out

def sample_train_indices(train_end_idx, rng, windows_per_series, lead_max=LEAD_MAX, l_past=L_PAST):
    "Return list of (sid, t_anchor, current_t, lead_pos). Subsampled per epoch."
    samples = []
    # current_t can be in [l_past + 1, train_end_idx - 1]; lead_pos in [1, lead_max] s.t. t_anchor >= l_past
    valid_current_lo = l_past + 1
    valid_current_hi = train_end_idx - 1
    if valid_current_hi - valid_current_lo + 1 <= 0:
        return samples
    for sid in range(N_SERIES):
        cur_ts = rng.integers(valid_current_lo, valid_current_hi + 1, size=windows_per_series)
        for cur_t in cur_ts:
            max_lead = min(lead_max, cur_t - l_past)
            lead = int(rng.integers(1, max_lead + 1))
            t_anchor = cur_t - lead
            samples.append((sid, t_anchor, int(cur_t), lead))
    return samples


class HybridTrainDataset(Dataset):
    def __init__(self, samples, scaled_tensors, sid_mu, sid_sd):
        self.samples = samples
        self.tens    = scaled_tensors        # dict sid → (T, n_feat) scaled
        self.sid_mu  = sid_mu                # for un-normalizing at inference; not needed here
        self.sid_sd  = sid_sd
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        sid, t_anchor, cur_t, lead = self.samples[i]
        arr = self.tens[sid]
        past = arr[t_anchor - L_PAST : t_anchor]                  # (L_PAST, n_feat)
        current_cov  = arr[cur_t, :N_COV]                          # (N_COV,)
        current_time = arr[cur_t, N_COV : N_COV + N_TIME]          # (N_TIME,)
        y_norm       = arr[cur_t, -1]                              # standardized target
        sid_mean     = (self.sid_mu[sid] - self.sid_mu[sid]) / self.sid_sd[sid]   # always 0 by construction
        # We pass the *un-scaled* series mean (in target's normalized scale) — which is 0 since target was
        # standardized per series. Instead pass the per-series μ as a tiny float, scaled by the same σ to
        # keep it in the model's working scale:
        sid_mean = np.float32(0.0)   # carries no info on its own — included for consistency with 7a-tabular
        return (
            torch.from_numpy(past),
            torch.from_numpy(current_cov),
            torch.from_numpy(current_time),
            torch.tensor([lead / LEAD_MAX], dtype=torch.float32),
            torch.tensor(sid, dtype=torch.long),
            torch.tensor([sid_mean], dtype=torch.float32),
            torch.tensor(y_norm, dtype=torch.float32),
        )


def train_hybrid_one_seed(scaled_tensors, sid_mu, sid_sd, train_end_idx, seed):
    "Train one seed; return (oof_pred_unnormalized, gate_values, best_val_loss)."
    torch.manual_seed(seed)
    np.random.seed(seed)
    rng = np.random.default_rng(seed)

    model = HybridModel(
        n_cov=N_COV, n_time=N_TIME, n_series=N_SERIES, embed_dim=EMBED_DIM,
        enc_hidden=ENC_HIDDEN, enc_heads=ENC_HEADS, enc_layers=ENC_LAYERS,
        enc_dropout=ENC_DROPOUT, max_past_len=L_PAST,
        head_hidden=HEAD_HIDDEN, head_dropout=HEAD_DROPOUT,
    ).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    loss_fn = nn.L1Loss()

    # Inner-val split: hold out 1 series' tail for early stopping. Cheaper than a row-level holdout
    # because it still lets the encoder learn from the full past of all series.
    val_sid = int(rng.integers(0, N_SERIES))
    val_cur_lo, val_cur_hi = L_PAST + 1, train_end_idx - 1
    val_cur_ts = rng.choice(np.arange(val_cur_lo, val_cur_hi + 1),
                            size=min(2000, val_cur_hi - val_cur_lo + 1), replace=False)

    best_val_loss, patience, best_state = float("inf"), 0, None

    for epoch in range(EPOCHS):
        # Resample training windows every epoch — fresh stochastic exposure to lead positions
        samples = sample_train_indices(train_end_idx, rng, WINDOWS_PER_SERIES_PER_EPOCH)
        ds = HybridTrainDataset(samples, scaled_tensors, sid_mu, sid_sd)
        loader = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=0, drop_last=False)

        model.train()
        for past, cur_cov, cur_time, lead, sid_b, s_mean, y in loader:
            past, cur_cov = past.to(DEVICE), cur_cov.to(DEVICE)
            cur_time, lead = cur_time.to(DEVICE), lead.to(DEVICE)
            sid_b, s_mean, y = sid_b.to(DEVICE), s_mean.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            pred = model(past, cur_cov, cur_time, lead, sid_b, s_mean)
            loss = loss_fn(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        # Inner-val: pick fixed val series + random current_t's. Eval L1 on standardized target.
        model.eval()
        val_loss = _eval_inner_val(model, scaled_tensors[val_sid], val_sid, val_cur_ts, sid_mu, sid_sd)
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state    = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience      = 0
        else:
            patience += 1
            if patience >= EARLY_STOP:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # OOF inference on test slice for this fold — return un-normalized predictions.
    oof_pred = _oof_predict(model, scaled_tensors, sid_mu, sid_sd, train_end_idx)
    gate_vals = model.gate.values()
    return oof_pred, gate_vals, best_val_loss


def _eval_inner_val(model, scaled_tensor, sid, cur_ts, sid_mu, sid_sd):
    "L1 loss in standardized target space — fast cheap signal for early stopping."
    model.eval()
    L = L_PAST
    pasts, covs, times, leads, sids, means, ys = [], [], [], [], [], [], []
    for cur_t in cur_ts:
        max_lead = min(LEAD_MAX, cur_t - L)
        if max_lead < 1: continue
        lead = max_lead
        t_anchor = cur_t - lead
        pasts.append(scaled_tensor[t_anchor - L : t_anchor])
        covs.append(scaled_tensor[cur_t, :N_COV])
        times.append(scaled_tensor[cur_t, N_COV : N_COV + N_TIME])
        leads.append([lead / LEAD_MAX])
        sids.append(sid)
        means.append([0.0])
        ys.append(scaled_tensor[cur_t, -1])
    if not pasts:
        return float("inf")
    past_t  = torch.tensor(np.stack(pasts),  dtype=torch.float32, device=DEVICE)
    cov_t   = torch.tensor(np.stack(covs),   dtype=torch.float32, device=DEVICE)
    time_t  = torch.tensor(np.stack(times),  dtype=torch.float32, device=DEVICE)
    lead_t  = torch.tensor(np.array(leads),  dtype=torch.float32, device=DEVICE)
    sid_t   = torch.tensor(sids,             dtype=torch.long,    device=DEVICE)
    mean_t  = torch.tensor(np.array(means),  dtype=torch.float32, device=DEVICE)
    y_t     = torch.tensor(np.array(ys),     dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        p = model(past_t, cov_t, time_t, lead_t, sid_t, mean_t)
    return float(torch.mean(torch.abs(p - y_t)))


def _oof_predict(model, scaled_tensors, sid_mu, sid_sd, train_end_idx, batch_size=512):
    "For each series, encode the fixed train-tail window once; run head over all EVAL_HORIZON test rows."
    model.eval()
    L = L_PAST
    oof = np.zeros((N_SERIES, EVAL_HORIZON), dtype=np.float32)
    for sid in range(N_SERIES):
        arr = scaled_tensors[sid]
        # Encoder input: last L_PAST rows of training slice.
        past = torch.tensor(arr[train_end_idx - L : train_end_idx][None, ...],
                            dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            past_ctx = model.past_enc(past)              # (1, ENC_HIDDEN)
            sid_e    = model.series_emb(torch.tensor([sid], device=DEVICE))
        # Test rows: train_end_idx .. train_end_idx + EVAL_HORIZON - 1
        test_rows = arr[train_end_idx : train_end_idx + EVAL_HORIZON]   # (336, n_feat)
        cov_t    = torch.tensor(test_rows[:, :N_COV],          dtype=torch.float32, device=DEVICE)
        time_t   = torch.tensor(test_rows[:, N_COV : N_COV + N_TIME], dtype=torch.float32, device=DEVICE)
        leads_v  = torch.tensor(np.arange(1, EVAL_HORIZON + 1)[:, None] / LEAD_MAX,
                                dtype=torch.float32, device=DEVICE)
        mean_t   = torch.zeros(EVAL_HORIZON, 1, dtype=torch.float32, device=DEVICE)
        ctx_rep  = past_ctx.expand(EVAL_HORIZON, -1)
        sid_rep  = sid_e.expand(EVAL_HORIZON, -1)
        with torch.no_grad():
            gated = model.gate(cov_t)
            x = torch.cat([gated, time_t, leads_v, sid_rep, mean_t, ctx_rep], dim=-1)
            pred_norm = model.head(x).squeeze(-1).cpu().numpy()
        oof[sid] = pred_norm * sid_sd[sid] + sid_mu[sid]
    # Clip at 0
    return np.clip(oof, 0.0, None)


print("Hybrid training utilities ready.")

Hybrid training utilities ready.


---
## §6. Cross-validation run — 3 folds × 3 seeds

Each fold: fit per-fold scalers, train hybrid model under each seed, average OOF predictions across seeds. Report per-fold WAPE and per-seed spread. Save the learned gate values from every seed — those are the §7 deliverable.

In [10]:
import time
code_to_sid = {v: k for k, v in sid_to_code.items()}

oof_hybrid_rows, per_fold_wape, per_fold_seed_spread = [], [], []
gate_vals_per_seed_fold = []   # list of (fold, seed, np.array of 11)

for k in range(N_FOLDS):
    fold_t0 = time.time()
    end_idx = fold_train_end_idx(k)
    print(f"\n── Fold {k}: train_end_idx = {end_idx}, test = [{end_idx}, {end_idx + EVAL_HORIZON}) ──")

    # Fit scalers per fold (leakage-free).
    sid_mu, sid_sd = fit_per_series_target_stats(series_tensors_full, end_idx)
    feat_scaler = fit_feature_scaler(series_tensors_full, end_idx)
    scaled = {sid: apply_scalers_to_tensor(arr, feat_scaler, sid_mu[sid], sid_sd[sid])
              for sid, arr in series_tensors_full.items()}

    # Train each seed, collect OOFs (un-normalized) and gate values.
    seed_oofs, seed_wapes = [], []
    for s in SEEDS:
        seed_t0 = time.time()
        oof_pred, gates, val_loss = train_hybrid_one_seed(scaled, sid_mu, sid_sd, end_idx, s)
        seed_oofs.append(oof_pred)
        gate_vals_per_seed_fold.append((k, s, gates))

        # Compute this-seed WAPE on this fold's test slice using true targets.
        test_true = np.stack([series_tensors_full[sid][end_idx : end_idx + EVAL_HORIZON, -1]
                              for sid in range(N_SERIES)], axis=0)
        seed_wape = wape(test_true.ravel(), oof_pred.ravel())
        seed_wapes.append(seed_wape)
        print(f"   seed {s:>4}: best_val_L1 = {val_loss:.4f},  WAPE = {seed_wape:.3f},  "
              f"time = {time.time() - seed_t0:.1f}s")

    avg_oof = np.mean(np.stack(seed_oofs, axis=0), axis=0)
    test_true = np.stack([series_tensors_full[sid][end_idx : end_idx + EVAL_HORIZON, -1]
                          for sid in range(N_SERIES)], axis=0)
    fold_wape = wape(test_true.ravel(), avg_oof.ravel())
    spread    = max(seed_wapes) - min(seed_wapes)
    per_fold_wape.append(fold_wape)
    per_fold_seed_spread.append(spread)
    print(f"   ── fold {k}: avg WAPE = {fold_wape:.3f}, seed spread = {spread:.3f}, "
          f"time = {time.time() - fold_t0:.1f}s")

    # Build per-row OOF DataFrame for downstream merging.
    test_ts = base_seq["timestamp"].to_numpy().reshape(N_SERIES, T_PER_SERIES)[:, end_idx : end_idx + EVAL_HORIZON]
    for sid in range(N_SERIES):
        oof_hybrid_rows.append(pd.DataFrame({
            "series_id": code_to_sid[sid],
            "timestamp": pd.to_datetime(test_ts[sid]),
            "target":    test_true[sid],
            "pred_hyb":  avg_oof[sid],
            "lead_h":    np.arange(1, EVAL_HORIZON + 1),
            "fold":      k,
        }))

oof_hybrid = pd.concat(oof_hybrid_rows, ignore_index=True)
hyb_arr = np.array(per_fold_wape)

print("\n=== Hybrid CV summary ===")
print(f"{'fold':>6}  {'WAPE':>8}  {'seed spread':>12}")
for k, (w, sp) in enumerate(zip(hyb_arr, per_fold_seed_spread)):
    print(f"  {k:>4}    {w:>7.3f}    {sp:>10.3f}")
print(f"  ────")
print(f"  mean    {hyb_arr.mean():>7.3f}")
print(f"  std     {hyb_arr.std():>7.3f}")

print(f"\nAnchors  — LGBM (§3): {lgbm_arr.mean():.3f}   |  MLP (7a): {BEATS_MLP_THRESHOLD:.2f}")
print(f"Hybrid solo (§6)     : {hyb_arr.mean():.3f}")
print(f"Δ vs LGBM            : {hyb_arr.mean() - lgbm_arr.mean():+.3f}")
print(f"Δ vs 7a-MLP          : {hyb_arr.mean() - BEATS_MLP_THRESHOLD:+.3f}")


── Fold 0: train_end_idx = 3312, test = [3312, 3648) ──
   seed   42: best_val_L1 = 0.2552,  WAPE = 14.952,  time = 92.5s
   seed  101: best_val_L1 = 0.2363,  WAPE = 15.195,  time = 67.6s
   seed  202: best_val_L1 = 0.3307,  WAPE = 16.288,  time = 55.4s
   ── fold 0: avg WAPE = 15.181, seed spread = 1.336, time = 215.8s

── Fold 1: train_end_idx = 3648, test = [3648, 3984) ──
   seed   42: best_val_L1 = 0.2606,  WAPE = 17.218,  time = 71.3s
   seed  101: best_val_L1 = 0.2692,  WAPE = 17.556,  time = 39.6s
   seed  202: best_val_L1 = 0.3245,  WAPE = 17.477,  time = 47.5s
   ── fold 1: avg WAPE = 17.060, seed spread = 0.338, time = 158.8s

── Fold 2: train_end_idx = 3984, test = [3984, 4320) ──
   seed   42: best_val_L1 = 0.2454,  WAPE = 15.935,  time = 118.8s
   seed  101: best_val_L1 = 0.2616,  WAPE = 16.057,  time = 75.3s
   seed  202: best_val_L1 = 0.3094,  WAPE = 16.008,  time = 63.6s
   ── fold 2: avg WAPE = 15.828, seed spread = 0.121, time = 258.0s

=== Hybrid CV summary ===
  f

---
## §7. The headline — learned feature gates and residual decorrelation

Two deliverables, both required by the framing of this experiment:

1. **Learned gate values per covariate**, averaged across seeds and folds, sorted descending. This is the model's answer to *"which forecasts do you actually need."*
2. **Pearson r against LGBM-MAE OOF residuals**, same rule as 7a/7b: `r < 0.85` → strong decorrelation.

In [11]:
# Gates table
gate_matrix = np.stack([g for (_, _, g) in gate_vals_per_seed_fold], axis=0)   # (N_FOLDS × N_SEEDS, N_COV)
gate_mean   = gate_matrix.mean(axis=0)
gate_std    = gate_matrix.std(axis=0)
order       = np.argsort(-gate_mean)

print("=== Learned feature gates (sigmoid scalar per covariate) ===")
print(f"  Averaged over {gate_matrix.shape[0]} (fold × seed) runs. 1.0 = full keep, 0.0 = full drop.\n")
print(f"  {'feature':<40}  {'mean':>6}  {'std':>6}  {'verdict':>10}")
print("  " + "-" * 70)
for i in order:
    name = DOMAIN_FEAT_COLS[i]
    v_mean, v_std = gate_mean[i], gate_std[i]
    if   v_mean > 0.70: verdict = "KEEP"
    elif v_mean > 0.40: verdict = "PARTIAL"
    else:               verdict = "DROP"
    print(f"  {name:<40}  {v_mean:>6.3f}  {v_std:>6.3f}  {verdict:>10}")

# Residual correlation diagnostic
joined = oof_lgbm.merge(
    oof_hybrid[["series_id", "timestamp", "pred_hyb", "fold"]],
    on=["series_id", "timestamp", "fold"], how="inner",
)
assert len(joined) == len(oof_lgbm) == len(oof_hybrid), \
    "OOF row counts diverged between LGBM and hybrid — investigate fold alignment."

joined["resid_lgbm"] = joined["target"] - joined["pred_lgbm"]
joined["resid_hyb"]  = joined["target"] - joined["pred_hyb"]

r_pooled = float(np.corrcoef(joined["resid_lgbm"], joined["resid_hyb"])[0, 1])
r_per_fold = [float(np.corrcoef(
    joined.loc[joined.fold == k, "resid_lgbm"],
    joined.loc[joined.fold == k, "resid_hyb"],
)[0, 1]) for k in range(N_FOLDS)]

print("\n=== Pearson r(residual_LGBM, residual_hybrid) ===")
print(f"{'fold':>6}  {'r':>7}")
for k, r in enumerate(r_per_fold):
    print(f"  {k:>4}    {r:>6.4f}")
print(f"  ────")
print(f"  pooled  {r_pooled:>6.4f}")

print("\n=== Verdict ===")
if hyb_arr.mean() <= BEATS_MLP_THRESHOLD:
    print(f"✅ Hybrid solo CV-WAPE {hyb_arr.mean():.3f} ≤ {BEATS_MLP_THRESHOLD:.2f} (7a-MLP). "
          "Real pure-DL contender; build the submission notebook on top of this.")
elif hyb_arr.mean() <= BEATS_LGBM_THRESHOLD:
    print(f"~  Hybrid solo {hyb_arr.mean():.3f} beats LGBM but not the 7a-MLP. "
          "The simpler MLP is the better pure-DL submission. Gate values are still the deliverable.")
else:
    print(f"✗  Hybrid solo {hyb_arr.mean():.3f} fails to beat LGBM. Architecture or training is broken — "
          "check the seed spread, val loss curve, and whether the encoder actually trained "
          "(do gate values move from 0.5?).")

if r_pooled < R_STRONG_DIVERSITY:
    print(f"   r = {r_pooled:.4f} < {R_STRONG_DIVERSITY} — also strongly decorrelated from LGBM. "
          "Could blend with LGBM in a future ensemble if needed.")
else:
    print(f"   r = {r_pooled:.4f} — residuals share structure with LGBM (expected; the input space overlaps).")

=== Learned feature gates (sigmoid scalar per covariate) ===
  Averaged over 9 (fold × seed) runs. 1.0 = full keep, 0.0 = full drop.

  feature                                     mean     std     verdict
  ----------------------------------------------------------------------
  queue_pressure_forecast                    0.572   0.010     PARTIAL
  maintenance_known                          0.562   0.012     PARTIAL
  network_pressure_forecast                  0.550   0.004     PARTIAL
  event_load_forecast                        0.534   0.002     PARTIAL
  promotion_intensity                        0.521   0.006     PARTIAL
  demand_forecast                            0.513   0.004     PARTIAL
  upstream_quality_forecast                  0.513   0.003     PARTIAL
  shock_risk                                 0.512   0.004     PARTIAL
  workload_intensity                         0.502   0.002     PARTIAL
  service_irregularity_risk_forecast         0.487   0.005     PARTIAL
  unit_relia

---
## §8. Per-horizon WAPE — did the past encoder help short-horizon predictions?

The 7a-seq2seq notebook showed the LSTM's only solo win was at lead-1h (WAPE 20.7 vs LGBM 24.1). If our past encoder is doing useful work, we should see a similar (or smaller) advantage at short horizons and parity at long horizons. If WAPE is flat and matches LGBM everywhere, the encoder isn't pulling signal that the per-row features don't already have.

In [12]:
BINS = [(1, 1), (2, 6), (7, 24), (25, 72), (73, 168), (169, 336)]

merged = oof_lgbm.merge(
    oof_hybrid[["series_id", "timestamp", "pred_hyb", "lead_h"]],
    on=["series_id", "timestamp"], how="inner",
)

print(f"{'lead':>10}  {'n':>6}  {'LGBM':>7}  {'hybrid':>7}  {'Δ_vs_LGBM':>10}")
print("-" * 50)
for lo, hi in BINS:
    sl = merged[(merged["lead_h"] >= lo) & (merged["lead_h"] <= hi)]
    if len(sl) == 0:
        continue
    w_l = wape(sl["target"], sl["pred_lgbm"])
    w_h = wape(sl["target"], sl["pred_hyb"])
    print(f"  {lo:>3}..{hi:<3}h  {len(sl):>6}  {w_l:>6.3f}  {w_h:>6.3f}  {w_l - w_h:>+9.3f}")

      lead       n     LGBM   hybrid   Δ_vs_LGBM
--------------------------------------------------
    1..1  h     288  24.052  25.156     -1.104
    2..6  h    1440  21.573  23.418     -1.845
    7..24 h    5184  12.514  14.170     -1.656
   25..72 h   13824  17.178  18.382     -1.204
   73..168h   27648  14.053  15.326     -1.273
  169..336h   48384  14.670  15.812     -1.142


---
## §9. Save OOF predictions + gate values, decision summary

Two artifacts:

- `oof_predictions_7d_hybrid.csv` — same schema as the other Phase 7 OOF CSVs. Ready to feed into a blend study if needed.
- `gates_7d_hybrid.csv` — per-(fold, seed) gate values for every covariate. The mean column is the headline finding for the professor.

In [13]:
oof_path = OUTPUT_DIR / "oof_predictions_7d_hybrid.csv"
oof_hybrid[["series_id", "timestamp", "target", "pred_hyb", "lead_h", "fold"]].to_csv(oof_path, index=False)
print(f"OOF predictions → {oof_path}")

gate_rows = []
for (k, s, gates) in gate_vals_per_seed_fold:
    for i, name in enumerate(DOMAIN_FEAT_COLS):
        gate_rows.append({"fold": k, "seed": s, "feature": name, "gate": float(gates[i])})
gate_df = pd.DataFrame(gate_rows)
# Add the per-feature mean column for easy reading
gate_summary = (gate_df.groupby("feature")["gate"]
                .agg(["mean", "std"])
                .reset_index()
                .sort_values("mean", ascending=False))
gates_path = OUTPUT_DIR / "gates_7d_hybrid.csv"
gate_summary.to_csv(gates_path, index=False)
print(f"Gate values → {gates_path}")
print()
print(gate_summary.to_string(index=False))

print("\n=== Decision summary ===")
print(f"  LGBM-MAE (§3)         : {lgbm_arr.mean():.3f} ± {lgbm_arr.std():.3f}")
print(f"  7a-tabular MLP (anchor): {BEATS_MLP_THRESHOLD:.2f}")
print(f"  Hybrid solo (§6)      : {hyb_arr.mean():.3f} ± {hyb_arr.std():.3f}")
print(f"  Pearson r vs LGBM (§7): {r_pooled:.4f}")
print(f"  Top-3 gates           : "
      + ", ".join([f"{DOMAIN_FEAT_COLS[i]} ({gate_mean[i]:.2f})" for i in order[:3]]))
print(f"  Bottom-3 gates        : "
      + ", ".join([f"{DOMAIN_FEAT_COLS[i]} ({gate_mean[i]:.2f})" for i in order[-3:]]))

OOF predictions → /kaggle/working/oof_predictions_7d_hybrid.csv
Gate values → /kaggle/working/gates_7d_hybrid.csv

                           feature     mean      std
           queue_pressure_forecast 0.571731 0.010349
                 maintenance_known 0.561543 0.012276
         network_pressure_forecast 0.549564 0.004677
               event_load_forecast 0.534074 0.002432
               promotion_intensity 0.520700 0.006505
                   demand_forecast 0.513293 0.004757
         upstream_quality_forecast 0.512822 0.003263
                        shock_risk 0.512318 0.004129
                workload_intensity 0.502098 0.001698
service_irregularity_risk_forecast 0.486903 0.005638
         unit_reliability_forecast 0.486132 0.005099

=== Decision summary ===
  LGBM-MAE (§3)         : 14.782 ± 0.616
  7a-tabular MLP (anchor): 14.52
  Hybrid solo (§6)      : 16.023 ± 0.779
  Pearson r vs LGBM (§7): 0.9696
  Top-3 gates           : queue_pressure_forecast (0.57), maintenance_known

---
## §10. Next step depending on the §9 result

- **Hybrid CV-WAPE ≤ 14.52**: the encoder and gates are extracting real signal. Build a 7e submission notebook (refit on full train, 5 seeds, predict on `validation_input.csv`). This is the pure-DL submission for the grade.
- **14.52 < hybrid CV-WAPE ≤ 14.78**: the hybrid additions cost more than they helped. The simpler 7a-tabular MLP is the better pure-DL submission. Repackage the 7a-tabular OOF + a refit-on-full-train inference step. Gate values from this notebook are still the professor deliverable.
- **Hybrid CV-WAPE > 14.78**: debugging time. Likely causes, in order: (a) inner-val on a single series is too noisy and early-stopping triggered too early — try `EARLY_STOP=10` or row-level holdout; (b) `WINDOWS_PER_SERIES_PER_EPOCH` too small — bump to 256; (c) encoder hidden too large for the data — drop to 64; (d) feature gates collapsing to 0 — check `gate_std` is small AND `gate_mean` is reasonable, not stuck at 0.5.
- **Gate values uniformly close to 1.0**: model didn't use the gating mechanism at all (it can't lose by setting all gates to 1, so this is the path of least resistance during training). Add an L1 penalty on `gates_logits` to force sparsity, or use a softmax-VSN instead of independent sigmoids.
- **Gate values dispersed (some near 1, some near 0)**: that's the answer the professor was asking for. The dropped ones can be removed from the feature set in any future model with no expected accuracy loss.